# 01 · Data exploration & corpus preparation (PRD §6)
**Runs:** locally (`.venv`). **Produces:** `data/processed/earbud_reviews.parquet`,
`data/processed/labeling_pool.parquet`.

1. Download the Electronics review + metadata files (~25 GB total, one time).
2. Filter to earbud products, clean text with the production Agent 1 cleaner,
   drop < 20-word and duplicate reviews.
3. Draw the star-stratified 2,000-review labeling pool (400 per star, ≤ 5 per product).

In [ ]:
# --- Repo setup: make `app` (backend) and `vera_training` importable ---------
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "PRD.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ("backend", "training"):
    if str(ROOT / sub) not in sys.path:
        sys.path.insert(0, str(ROOT / sub))

import os
if not (ROOT / ".env").exists():  # app settings require these; real values come from .env
    os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
    os.environ.setdefault("REDIS_URL", "redis://localhost:6379/0")

from vera_training import paths
paths.ensure_dirs()
print("Repo root:", ROOT)

## 1. Download raw files
The files below are the raw JSONL exports in the dataset repo. Confirm the paths on
https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023/tree/main/raw
before running. Skipped automatically if the files already exist.

In [ ]:
from huggingface_hub import hf_hub_download

FILES = {
    "meta_Electronics.jsonl": "raw/meta_categories/meta_Electronics.jsonl",
    "Electronics.jsonl": "raw/review_categories/Electronics.jsonl",
}
for local_name, repo_path in FILES.items():
    target = paths.RAW / local_name
    if target.exists():
        print("exists:", target)
        continue
    downloaded = hf_hub_download(
        "McAuley-Lab/Amazon-Reviews-2023", repo_path, repo_type="dataset", local_dir=paths.RAW
    )
    Path(downloaded).rename(target)
    print("downloaded:", target)

## 2. Filter + clean (streams the 20+ GB review file; ~20–40 min)

In [ ]:
from vera_training.data import json_records, prepare_corpus

if paths.CORPUS.exists():
    import pandas as pd
    corpus = pd.read_parquet(paths.CORPUS)
else:
    corpus = prepare_corpus(
        json_records(paths.RAW / "Electronics.jsonl"),
        json_records(paths.RAW / "meta_Electronics.jsonl"),
        min_words=20,
    )
    corpus.to_parquet(paths.CORPUS, index=False)
print(f"{len(corpus):,} earbud reviews across {corpus.parent_asin.nunique():,} products")

## 3. Exploration — how inflated are the stars?

In [ ]:
import matplotlib.pyplot as plt

share = corpus.rating.value_counts(normalize=True).sort_index()
print((share * 100).round(1).astype(str) + "%")
share.plot.bar(title="Star distribution (earbuds)", ylabel="share", xlabel="stars")
plt.show()

reviews_per_product = corpus.parent_asin.value_counts()
print(reviews_per_product.describe())
print("Products with exactly 1 review:", (reviews_per_product == 1).sum())

## 4. Labeling pool (PRD §6.3)

In [ ]:
from vera_training.data import sample_for_labeling

pool = sample_for_labeling(corpus, n=2000, seed=42, product_cap=5)
pool.to_parquet(paths.LABELING_POOL, index=False)
print(pool.rating.value_counts().sort_index())
print("max reviews per product:", pool.parent_asin.value_counts().max())